In [14]:
import re
import sys
import csv
import json
import subprocess
from pathlib import Path
from collections import defaultdict, Counter

In [15]:
def find_repo_root(marker: str = 'dataset', start: Path | None = None) -> Path:
    """Searches upward from the current working directory until a folder
    named 'marker' is found -- robust against the kernel's working directory
    not matching the notebook's own folder."""
    start = start or Path.cwd()

    for parent in [start, *start.parents]:
        if (parent / marker).is_dir():
            return parent

    raise FileNotFoundError(
        f"Could not find a folder named '{marker}' above {start}. "
        f'Current working directory: {Path.cwd()}'
    )


REPO_ROOT = find_repo_root()

CODE_DIR_PATH = REPO_ROOT / 'dataset' / 'storybook' / 'src' / 'code'

# 'components' or 'uis'
TYPE = 'uis'
TYPE_DIR_PATH = CODE_DIR_PATH / TYPE

COMPLEXITIES = ['simple', 'medium', 'hard']
VARIANTS = ['pretty', 'messy']
APPROACHES = ['gt', 'a', 'b', 'c', 'd']

# Directory structure
# for components
#   {TYPE}/gt/{complexity}/{1-10}.vue
#   {TYPE}/a/{complexity}/{file}.vue
#   {TYPE}/{b|c|d}/{prompt_strategy}/{complexity}/{file}.vue
# for uis
#   {TYPE}/gt/{1-5}.vue                          <- FLAT, no complexity/variant subfolder
#   {TYPE}/a/{variant}/{1-5}-a.vue                <- variant, but NO prompt_strategy level
#   {TYPE}/{b|c|d}/{prompt_strategy}/{variant}/{file}.vue

# Generic name for whichever second-level grouping applies for the current
# TYPE -- a complexity level for 'components', a pretty/messy variant for
# 'uis'. Used throughout instead of hardcoding COMPLEXITIES, so the same
# loading/aggregation code works for both directory layouts.
GROUP_LEVELS = COMPLEXITIES if TYPE == 'components' else VARIANTS
EXPECTED_GROUPS = set(GROUP_LEVELS)

# Approach 'a' is rule-based/deterministic and genuinely has NO prompt
# strategy -- not even a conceptual one, unlike B/C/D. It is still stored in
# the same 3-level {prompt_strategy}/{group}/{stem} structure as B/C/D below,
# but under the real value None (not an invented label like 'deterministic'),
# consistent with storybook-generate-stories.ipynb. None works fine as a
# dict key here and avoids having to special-case 'a' anywhere downstream
# (aggregation, CSV export) that already iterates "whatever prompt_strategy
# keys exist".
A_PROMPT_STRATEGY_KEY = None

# Placeholder 'group' key for GT under TYPE == 'uis', which has no
# complexity/variant subfolder at all (GT is variant-independent, just like
# the Figma JSON/screenshot it comes from).
GT_FLAT_GROUP_KEY = '_flat'

PRIMEVUE_COMPONENTS = {
    'Accordion', 'AccordionPanel', 'AccordionHeader', 'AccordionContent',
    'Avatar', 'AvatarGroup',
    'Badge', 'Breadcrumb', 'Button',
    'Card', 'Checkbox', 'Column', 'ColumnGroup',
    'DataTable', 'DatePicker', 'Dialog', 'Divider',
    'InputNumber', 'InputText',
    'Menu',
    'OverlayBadge',
    'Password', 'Popover', 'ProgressBar',
    'RadioButton', 'Row',
    'Select', 'Skeleton', 'Slider',
    'Tab', 'TabList', 'TabPanel', 'TabPanels', 'Tabs', 'Tag', 'Textarea',
    'ToggleSwitch',
    'IconField', 'InputIcon',
}
PRIMEVUE_COMPONENTS_NORM = {c.lower(): c for c in PRIMEVUE_COMPONENTS}

SKIP_PROPS = {'class', 'style', 'id', 'ref', 'key'}

API_CATALOG_PATH = REPO_ROOT / 'primevue' / 'component-types' / 'primevue-api-types-v1.json'

if API_CATALOG_PATH.exists():
    API_CATALOG: dict = json.loads(API_CATALOG_PATH.read_text(encoding='utf-8'))
    print(f'API catalog loaded: {len(API_CATALOG)} components from {API_CATALOG_PATH}')
else:
    API_CATALOG = {}
    print(f'WARNING: {API_CATALOG_PATH} not found -- PrimeVue API Conformity (step 3) '
          f'will return None for all mockups. Run build_primevue_api_catalog.ipynb first.')

GENERIC_PROPS = {'pt', 'ptOptions', 'dt', 'unstyled'}

# vue-tsc setup for Compile Success Rate (step 1) -- must point to a Vue project
# with PrimeVue installed and a matching tsconfig.json.
VUE_PROJECT_ROOT = REPO_ROOT / 'dataset' / 'storybook'
VUE_TSC_AVAILABLE = (VUE_PROJECT_ROOT / 'node_modules' / '.bin' / 'vue-tsc').exists() or \
                     (VUE_PROJECT_ROOT / 'node_modules' / '.bin' / 'vue-tsc.cmd').exists()

print(f'vue-tsc available under {VUE_PROJECT_ROOT}: {VUE_TSC_AVAILABLE}')
if not VUE_TSC_AVAILABLE:
    print('  -> Compile Success Rate will return None for all mockups until vue-tsc is available.')

API catalog loaded: 36 components from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\primevue\component-types\primevue-api-types-v1.json
vue-tsc available under C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook: True


## 1. Load reference and generated code files in groups by method, prompt strategy and complexity

In [16]:
CODE_FILES_BY_APPROACH: dict = {
    'gt': defaultdict(dict),
    'a':  defaultdict(lambda: defaultdict(dict)),
    'b':  defaultdict(lambda: defaultdict(dict)),
    'c':  defaultdict(lambda: defaultdict(dict)),
    'd':  defaultdict(lambda: defaultdict(dict)),
}

for approach in APPROACHES:
    approach_dir = TYPE_DIR_PATH / approach

    print(f'Loading code files for approach {approach} from {approach_dir}')

    if not approach_dir.exists():
        print(f'    Skip (not existing): {approach_dir}')

        continue

    if approach == 'gt':
        if TYPE == 'components':
            for complexity_dir in approach_dir.iterdir():
                if not complexity_dir.is_dir() or complexity_dir.name not in EXPECTED_GROUPS:
                    continue

                complexity = complexity_dir.name

                for vue_file in complexity_dir.glob('*.vue'):
                    content = vue_file.read_text(encoding='utf-8', errors='ignore')

                    if vue_file.stem in CODE_FILES_BY_APPROACH['gt'][complexity]:
                        print(f'  WARNING: Duplicate file {vue_file.stem} in {complexity} for gt, overwriting previous content.')

                    CODE_FILES_BY_APPROACH['gt'][complexity][vue_file.stem] = content

        else:
            # uis: gt/{1-5}.vue -- flat, GT is variant-independent (same file
            # used as reference for both pretty- and messy-derived generations).
            for vue_file in approach_dir.glob('*.vue'):
                content = vue_file.read_text(encoding='utf-8', errors='ignore')

                if vue_file.stem in CODE_FILES_BY_APPROACH['gt'][GT_FLAT_GROUP_KEY]:
                    print(f'  WARNING: Duplicate file {vue_file.stem} for gt, overwriting previous content.')

                CODE_FILES_BY_APPROACH['gt'][GT_FLAT_GROUP_KEY][vue_file.stem] = content

    elif approach == 'a':
        # components: a/{complexity}/{file}.vue
        # uis:        a/{variant}/{file}.vue
        # No prompt_strategy level either way -- stored under A_PROMPT_STRATEGY_KEY
        # (real None, see config cell) so the rest of the notebook can treat
        # 'a' uniformly with b/c/d without special-casing it.
        for group_dir in approach_dir.iterdir():
            if not group_dir.is_dir() or group_dir.name not in EXPECTED_GROUPS:
                continue

            group = group_dir.name

            for vue_file in group_dir.glob('*.vue'):
                content = vue_file.read_text(encoding='utf-8', errors='ignore')

                if vue_file.stem in CODE_FILES_BY_APPROACH['a'][A_PROMPT_STRATEGY_KEY][group]:
                    print(f'  WARNING: Duplicate file {vue_file.stem} in {group} for a, overwriting previous content.')

                CODE_FILES_BY_APPROACH['a'][A_PROMPT_STRATEGY_KEY][group][vue_file.stem] = content

    else:
        for strategy_dir in approach_dir.iterdir():
            if not strategy_dir.is_dir():
                continue

            prompt_strategy = strategy_dir.name

            for group_dir in strategy_dir.iterdir():
                if not group_dir.is_dir() or group_dir.name not in EXPECTED_GROUPS:
                    continue

                group = group_dir.name

                for vue_file in group_dir.glob('*.vue'):
                    content = vue_file.read_text(encoding='utf-8', errors='ignore')

                    if vue_file.stem in CODE_FILES_BY_APPROACH[approach][prompt_strategy][group]:
                        print(f'  WARNING: Duplicate file {vue_file.stem} in {group} for {approach}/{prompt_strategy}, overwriting previous content.')

                    CODE_FILES_BY_APPROACH[approach][prompt_strategy][group][vue_file.stem] = content


print('\nCode files loaded:')
for approach, data in CODE_FILES_BY_APPROACH.items():
    if approach == 'gt':
        total_files = sum(len(files) for files in data.values())

        print(f'Approach {approach}: {total_files} files')
    else:
        total_files = sum(len(files) for strategies in data.values() for files in strategies.values())

        print(f'Approach {approach}: {total_files} files')

        for strategy, groups in data.items():
            strategy_total = sum(len(files) for files in groups.values())
            strategy_label = strategy if strategy is not None else '(none -- deterministic, approach a)'

            print(f'  Strategy {strategy_label}: {strategy_total} files')

            for group, files in groups.items():
                print(f'    Group {group}: {len(files)} files')

Loading code files for approach gt from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\code\uis\gt
Loading code files for approach a from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\code\uis\a
Loading code files for approach b from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\code\uis\b
Loading code files for approach c from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\code\uis\c
Loading code files for approach d from C:\Users\chris\Nextcloud\Studium\Masterarbeit\-Repo\dataset\storybook\src\code\uis\d

Code files loaded:
Approach gt: 5 files
Approach a: 10 files
  Strategy (none -- deterministic, approach a): 10 files
    Group messy: 5 files
    Group pretty: 5 files
Approach b: 180 files
  Strategy few_shot: 90 files
    Group messy: 45 files
    Group pretty: 45 files
  Strategy zero_shot: 90 files
    Group messy: 45 files
    Group pretty: 45 files
Approach c: 180 fil

## 2. Basis-Extraction

In [17]:
def extract_template_block(sfc: str) -> str | None:
    """Finds the outer <template> block, robust against nested named-slot
    templates (<template #slotname>...</template>)."""
    start = re.search(r'<template>', sfc)

    if not start:
        return None

    pos = start.end()
    depth = 1

    for m in re.finditer(r'<template(?:\s[^>]*)?(/?)>|</template>', sfc[pos:]):
        token = m.group(0)

        if token == '</template>':
            depth -= 1

            if depth == 0:
                return sfc[pos: pos + m.start()]

        elif not token.endswith('/>'):
            depth += 1

    return sfc[pos:]


def extract_script_block(sfc: str) -> str | None:
    """Finds the <script ...>...</script> block. Takes the FIRST block if
    several are present (e.g. <script> + <script setup> side by side) --
    our pipeline exclusively generates <script setup>."""
    m = re.search(r'<script[^>]*>(.*?)</script>', sfc, re.DOTALL)

    return m.group(1) if m else None


def extract_components_detailed(sfc: str) -> dict:
    """As in evaluation_f1_scores.ipynb: recognized/unrecognized_components/native_elements."""
    template = extract_template_block(sfc)

    recognized: list[str] = []
    unrecognized_components: Counter = Counter()
    native_elements: Counter = Counter()

    if template is None:
        return {'recognized': recognized, 'unrecognized_components': unrecognized_components,
                'native_elements': native_elements}

    for m in re.finditer(r'<([A-Za-z][A-Za-z0-9]*)', template):
        tag = m.group(1)
        canonical = PRIMEVUE_COMPONENTS_NORM.get(tag.lower())

        if canonical:
            recognized.append(canonical)
        elif tag[0].isupper():
            unrecognized_components[tag] += 1
        else:
            native_elements[tag] += 1

    return {'recognized': recognized, 'unrecognized_components': unrecognized_components,
            'native_elements': native_elements}


def extract_components(sfc: str) -> list[str]:
    return extract_components_detailed(sfc)['recognized']


_LITERAL_RE = re.compile(r"^\s*(true|false|-?\d+(\.\d+)?|'[^']*'|\"[^\"]*\")\s*$", re.IGNORECASE)


def is_literal_expr(expr: str) -> bool:
    return bool(_LITERAL_RE.match(expr.strip()))


def normalize_prop_value(raw: str) -> str:
    if raw == '__boolean__':
        return 'true'

    v = raw.strip()

    if (v.startswith('"') and v.endswith('"')) or (v.startswith("'") and v.endswith("'")):
        v = v[1:-1]

    low = v.lower()
    if low in ('true', 'false'):
        return low

    try:
        return repr(float(v))
    except ValueError:
        pass

    return v


def parse_props(attrs_str: str) -> dict[str, str]:
    """As in evaluation_prop_accuracy.ipynb: static, dynamic, v-model, boolean shorthand."""
    props: dict[str, str] = {}

    if not attrs_str:
        return props

    for m in re.finditer(r'v-model(?::([a-zA-Z][\w-]*))?="([^"]*)"', attrs_str):
        named_prop, val = m.group(1), m.group(2)
        key = f':{named_prop}' if named_prop else ':modelValue'
        props[key] = val

    for m in re.finditer(r':([a-zA-Z][\w-]*)="([^"]*)"', attrs_str):
        key, val = m.group(1), m.group(2)

        if key not in SKIP_PROPS and not key.startswith('v-'):
            props[f':{key}'] = val

    for m in re.finditer(r'(?<!:)\b([a-zA-Z][\w-]*)="([^"]*)"', attrs_str):
        key, val = m.group(1), m.group(2)

        if key not in SKIP_PROPS and not key.startswith(('v-', '@')):
            props[key] = val

    stripped = re.sub(r'(?:@|:)?[a-zA-Z][\w-]*(?::[a-zA-Z][\w-]*)?="[^"]*"', ' ', attrs_str)

    for m in re.finditer(r'\b([a-zA-Z][\w-]*)\b', stripped):
        key = m.group(1)

        if key not in SKIP_PROPS and not key.startswith('v-') \
           and key not in props and key[0].islower():
            props[key] = '__boolean__'

    return props


_TAG_RE = re.compile(r'<([A-Za-z][A-Za-z0-9]*)((?:\s[^>]*?)?)\s*(/?)>')


def extract_component_instances(sfc: str) -> dict[str, list[dict[str, str]]]:
    """As in evaluation_prop_accuracy.ipynb: all instances of known PrimeVue
    components together with their props, in document order, recognized case-insensitively."""
    template = extract_template_block(sfc)
    instances: dict[str, list[dict[str, str]]] = {}

    if template is None:
        return instances

    for m in _TAG_RE.finditer(template):
        tag, attrs_str = m.group(1), m.group(2)
        canonical = PRIMEVUE_COMPONENTS_NORM.get(tag.lower())

        if canonical is None:
            continue

        instances.setdefault(canonical, []).append(parse_props(attrs_str))

    return instances


def is_static_bindable(type_str: str | None) -> bool:
    """Heuristic for Binding Correctness: only 'string'/'HintedString<...>' are
    considered statically bindable (see build_primevue_api_catalog.ipynb)."""
    if type_str is None:
        return False

    t = type_str.replace('| undefined', '').replace('|undefined', '').strip()

    return t == 'string' or t.startswith('HintedString<')

## 3. Step 0 - Parse-OK-Rate

In [18]:
def compute_parse_ok(sfc: str) -> bool:
    """Share of outputs with a valid <template> and <script> block --
    here as a binary value per file, aggregation as a rate happens later."""
    return '<template>' in sfc and '<script' in sfc

## 4. Step 1 - Compile Success Rate

In [19]:
_TMP_COMPILE_DIR = VUE_PROJECT_ROOT / 'src' / '_eval_tmp'


def _resolve_vue_tsc_bin() -> Path:
    """On Windows, npm installs CLI tools like vue-tsc as a .cmd batch file
    (node_modules/.bin/vue-tsc.cmd), NOT as a native .exe -- the extension-
    less 'vue-tsc' file that also exists there is a Unix shebang shell script
    (plain text, not a valid Win32 binary). Launching that file directly via
    subprocess on Windows raises WinError 193 ('%1 ist keine zulaessige
    Win32-Anwendung') because CreateProcess refuses to execute a non-PE file.
    VUE_TSC_AVAILABLE already checks for both variants; this picks the one
    that is actually launchable on the current platform.
    """
    if sys.platform == 'win32':
        return VUE_PROJECT_ROOT / 'node_modules' / '.bin' / 'vue-tsc.cmd'

    return VUE_PROJECT_ROOT / 'node_modules' / '.bin' / 'vue-tsc'


def compute_compile_success(sfc: str) -> bool | None:
    """True/False if vue-tsc ran, None if vue-tsc is unavailable or the
    call itself fails (e.g. timeout, missing tsconfig)."""
    if not VUE_TSC_AVAILABLE:
        return None

    _TMP_COMPILE_DIR.mkdir(parents=True, exist_ok=True)
    tmp_file = _TMP_COMPILE_DIR / 'CompileCheck.vue'
    tmp_file.write_text(sfc, encoding='utf-8')

    vue_tsc_bin = _resolve_vue_tsc_bin()

    try:
        result = subprocess.run(
            [str(vue_tsc_bin), '--noEmit', '--ignoreConfig', str(tmp_file)],
            cwd=VUE_PROJECT_ROOT,
            capture_output=True,
            text=True,
            timeout=60,
            # .cmd files on Windows are batch scripts, not PE binaries --
            # CreateProcess needs cmd.exe as the interpreter to run them
            # reliably; shell=True routes the call through it. POSIX is
            # unaffected (vue-tsc there is a real executable shell script).
            shell=(sys.platform == 'win32'),
        )

        print(f'  vue-tsc returncode={result.returncode}, stdout={result.stdout.strip()}, stderr={result.stderr.strip()}')

        return result.returncode == 0

    except (subprocess.TimeoutExpired, OSError) as e:
        print(f'  WARNING: vue-tsc call failed ({e}) -- returning None instead of False.')

        return None
    finally:
        tmp_file.unlink(missing_ok=True)

## 5. Step 2 - Import-Completeness

In [20]:
def compute_import_completeness(sfc: str) -> float | None:
    """Share of PrimeVue components used in the template that are imported
    in the script. None if no PrimeVue component is used in the template
    (nothing to import -- not a meaningful value, not 1.0/0.0)."""
    used = set(extract_components(sfc))

    if not used:
        return None

    script = extract_script_block(sfc) or ''
    imported = {
        PRIMEVUE_COMPONENTS_NORM.get(m.group(1).lower(), m.group(1))
        for m in re.finditer(r"import\s+(\w+)\s+from\s+'primevue/", script)
    }

    return len(used & imported) / len(used)

## 6. Step 3 - PrimeVue-API-Conformity

In [21]:
def extract_pt_top_level_keys(template: str) -> list[str]:
    """Finds the top-level keys (first nesting level) for every :pt="{...}"
    occurrence, robust against nested object literals such as
    {root: {class: 'foo'}} -- there, only 'root' counts, not the nested
    'class'. A simple regex without depth counting would fail to match at
    all for ANY nested object, since '[^}]*' cannot read past the inner
    closing brace.
    """
    keys: list[str] = []

    for m in re.finditer(r':pt="\{', template):
        start = m.end() - 1
        depth = 1
        i = start + 1

        while i < len(template) and depth > 0:
            if template[i] == '{':
                depth += 1
            elif template[i] == '}':
                depth -= 1
            i += 1

        value = template[start + 1:i - 1]

        entries, pdepth, current = [], 0, []
        for ch in value:
            if ch in '{[(':
                pdepth += 1
            elif ch in '}])':
                pdepth -= 1
            elif ch == ',' and pdepth == 0:
                entries.append(''.join(current))
                current = []

                continue
            current.append(ch)

        tail = ''.join(current).strip()
        if tail:
            entries.append(tail)

        for entry in entries:
            km = re.match(r'\s*([a-zA-Z_]\w*)\s*:', entry)

            if km:
                keys.append(km.group(1))

    return keys


_SLOT_TEMPLATE_RE = re.compile(r'<template\s+#([\w-]+)')


def compute_api_conformity(sfc: str) -> dict:
    """Computes the raw counts for all four sub-metrics across all generated
    instances whose component type is in API_CATALOG -- independent of
    Component-F1 matching (see evaluation.md, UF2 step 3).

    Slot and Passthrough Correctness cannot be robustly attributed to a
    SINGLE instance without a full DOM parser with scope tracking (a slot
    template can be a child of any parent element). Approximation: a
    slot/PT key is considered valid if it exists for AT LEAST ONE of the
    catalog components used in the mockup -- a documented simplification,
    see limitations in evaluation.md.

    Returns: raw counts (correct, total) per sub-metric, so that aggregation
    across multiple mockups can be done as a micro-average (sum before division).
    """
    instances = extract_component_instances(sfc)

    prop_correct = prop_total = 0
    binding_correct = binding_total = 0

    comps_in_catalog = [c for c in instances if c in API_CATALOG]

    for comp in comps_in_catalog:
        catalog_entry = API_CATALOG[comp]
        catalog_props = set(catalog_entry.get('props', {}).keys()) | GENERIC_PROPS
        prop_types = catalog_entry.get('props', {})

        for props in instances[comp]:
            for key, value in props.items():
                is_dynamic = key.startswith(':')
                bare_key = key[1:] if is_dynamic else key

                prop_total += 1
                if bare_key in catalog_props:
                    prop_correct += 1

                type_str = prop_types.get(bare_key)
                if type_str is not None:
                    binding_total += 1

                    if is_dynamic or is_static_bindable(type_str):
                        binding_correct += 1

    valid_slots: set[str] = set()
    valid_pt_keys: set[str] = set()
    for comp in comps_in_catalog:
        valid_slots |= set(API_CATALOG[comp].get('slots', []))
        valid_pt_keys |= set(API_CATALOG[comp].get('pt_keys', []))

    template = extract_template_block(sfc) or ''

    used_slots = _SLOT_TEMPLATE_RE.findall(template)
    slot_correct = sum(1 for s in used_slots if s in valid_slots)
    slot_total = len(used_slots)

    used_pt_keys = extract_pt_top_level_keys(template)
    pt_correct = sum(1 for k in used_pt_keys if k in valid_pt_keys)
    pt_total = len(used_pt_keys)

    return {
        'prop_correct': prop_correct, 'prop_total': prop_total,
        'binding_correct': binding_correct, 'binding_total': binding_total,
        'slot_correct': slot_correct, 'slot_total': slot_total,
        'pt_correct': pt_correct, 'pt_total': pt_total,
        'n_instances_in_catalog': sum(len(instances[c]) for c in comps_in_catalog),
    }

## 7. Step 4 - Maintainability (AVU, CDR, LOC, CD)

In [22]:
_ARBITRARY_VALUE_RE = re.compile(r'-\[[^\]]+\]')


def compute_avu(sfc: str) -> float | None:
    """Arbitrary Value Usage: share of class tokens with arbitrary value
    syntax (w-[123px], bg-[#ff0000]) among all class tokens in the class attribute."""
    template = extract_template_block(sfc)

    if template is None:
        return None

    class_values = re.findall(r'class="([^"]*)"', template)
    tokens = [t for cv in class_values for t in cv.split()]

    if not tokens:
        return None

    arbitrary = sum(1 for t in tokens if _ARBITRARY_VALUE_RE.search(t))

    return arbitrary / len(tokens)


def compute_cdr(text: str, block_size: int = 3) -> float | None:
    """Code Duplication Rate -- Python-native substitute for token-based
    clone-detection tools (e.g. jscpd): a sliding window of block_size lines;
    lines that are part of a window occurring more than once count as duplicated.

    Limitation: line-based (sensitive to formatting), not token-/AST-based
    like a real clone-detection tool -- see evaluation.md.
    """
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    n = len(lines)

    if n < block_size:
        return None

    windows = [tuple(lines[i:i + block_size]) for i in range(n - block_size + 1)]
    counts = Counter(windows)

    duplicated_idx: set[int] = set()
    for i, w in enumerate(windows):
        if counts[w] > 1:
            duplicated_idx.update(range(i, i + block_size))

    return len(duplicated_idx) / n


def compute_loc(sfc: str) -> dict:
    """Physical line count, separately for template and script."""
    template = extract_template_block(sfc) or ''
    script = extract_script_block(sfc) or ''

    return {
        'loc_template': template.count('\n') + (1 if template.strip() else 0),
        'loc_script': script.count('\n') + (1 if script.strip() else 0),
        'loc_total': sfc.count('\n') + 1,
    }


_COMMENT_RE = re.compile(r'^\s*(//|/\*|<!--)')


def compute_comment_density(sfc: str) -> float | None:
    """Share of comment lines among the total line count. Line-based (a line
    counts as a comment line if it starts with a comment prefix AFTER trimming) --
    does not capture inline comments at the end of a line."""
    lines = sfc.splitlines()

    if not lines:
        return None

    comment_lines = sum(1 for l in lines if _COMMENT_RE.match(l))

    return comment_lines / len(lines)

## 8. File-Naming-Parsing

In [23]:
def parse_generated_stem(stem: str, approach: str) -> dict | None:
    parts = stem.split('-')

    if not parts[0].isdigit():
        return None

    index = parts[0].zfill(2)

    if approach == 'a':
        if len(parts) == 2 and parts[1] == 'a':
            return {'index': index, 'strategy': 'a', 'model': None, 'run': None}
        return None

    if len(parts) >= 4 and re.match(rf'^{approach}[123]$', parts[1]) and parts[-1].isdigit():
        strategy = parts[1]
        model = '-'.join(parts[2:-1])
        run = parts[-1]

        return {'index': index, 'strategy': strategy, 'model': model, 'run': run}

    return None


def gt_index(stem: str) -> str:
    return stem.zfill(2) if stem.isdigit() else stem

## 9. Main-Loop

In [24]:
results: list[dict] = []

for group in GROUP_LEVELS:
    # GT is variant-independent for 'uis' (flat, single set of files shared by
    # both pretty and messy) but genuinely per-complexity for 'components' --
    # look it up accordingly rather than assuming a per-group GT set always exists.
    gt_group_key = GT_FLAT_GROUP_KEY if TYPE == 'uis' else group
    gt_files = CODE_FILES_BY_APPROACH['gt'].get(gt_group_key, {})
    gt_by_index = {gt_index(stem): content for stem, content in gt_files.items()}
    gt_loc_by_index = {idx: compute_loc(sfc)['loc_total'] for idx, sfc in gt_by_index.items()}

    for approach in ['a', 'b', 'c', 'd']:
        approach_data = CODE_FILES_BY_APPROACH[approach]

        for prompt_strategy, by_group in approach_data.items():
            gen_files = by_group.get(group, {})

            for stem, gen_sfc in gen_files.items():
                parsed = parse_generated_stem(stem, approach)

                if parsed is None:
                    print(f'  WARNING: File name does not match any pattern: '
                          f'{approach}/{prompt_strategy}/{group}/{stem}')
                    continue

                gt_loc = gt_loc_by_index.get(parsed['index'])

                # Step 0
                parse_ok = compute_parse_ok(gen_sfc)

                # Step 1 (conditional on step 0)
                compile_ok = compute_compile_success(gen_sfc) if parse_ok else None

                # Step 2 (conditional on step 0)
                import_completeness = compute_import_completeness(gen_sfc) if parse_ok else None

                # Step 3 (conditional on step 0)
                conformity = compute_api_conformity(gen_sfc) if parse_ok else {
                    'prop_correct': 0, 'prop_total': 0, 'binding_correct': 0, 'binding_total': 0,
                    'slot_correct': 0, 'slot_total': 0, 'pt_correct': 0, 'pt_total': 0,
                    'n_instances_in_catalog': 0,
                }

                # Step 4 (conditional on step 0)
                if parse_ok:
                    avu = compute_avu(gen_sfc)
                    cdr = compute_cdr(gen_sfc)
                    loc = compute_loc(gen_sfc)
                    cd = compute_comment_density(gen_sfc)
                else:
                    avu = cdr = cd = None
                    loc = {'loc_template': None, 'loc_script': None, 'loc_total': None}

                loc_ratio = (loc['loc_total'] / gt_loc) if (loc['loc_total'] and gt_loc) else None

                print(f'{group}/{parsed["index"]} '
                      f'[{approach}/{prompt_strategy or "-"}/{parsed["strategy"]}/{parsed["model"] or "-"}]  '
                      f'parse_ok={parse_ok}  compile_ok={compile_ok}  '
                      f'import_compl={import_completeness}')

                results.append({
                    'mockup':          f'{group}-{parsed["index"]}',
                    'complexity':       group,  # complexity for 'components', variant for 'uis'
                    'index':            parsed['index'],
                    'approach':         approach,
                    'strategy':         parsed['strategy'],
                    'prompt_strategy':  prompt_strategy,
                    'model':            parsed['model'] or '',
                    'run':              parsed['run'] or '',

                    'parse_ok':         parse_ok,
                    'compile_ok':       compile_ok,
                    'import_completeness': import_completeness,

                    'prop_correct':     conformity['prop_correct'],
                    'prop_total':       conformity['prop_total'],
                    'binding_correct':  conformity['binding_correct'],
                    'binding_total':    conformity['binding_total'],
                    'slot_correct':     conformity['slot_correct'],
                    'slot_total':       conformity['slot_total'],
                    'pt_correct':       conformity['pt_correct'],
                    'pt_total':         conformity['pt_total'],

                    'avu':              avu,
                    'cdr':              cdr,
                    'loc_template':     loc['loc_template'],
                    'loc_script':       loc['loc_script'],
                    'loc_total':        loc['loc_total'],
                    'loc_ratio_to_gt':  loc_ratio,
                    'comment_density':  cd,
                })

print(f'\nComputed: {len(results)} mockup results')

  vue-tsc returncode=0, stdout=, stderr=
pretty/01 [a/-/a/-]  parse_ok=True  compile_ok=True  import_compl=1.0
  vue-tsc returncode=0, stdout=, stderr=
pretty/02 [a/-/a/-]  parse_ok=True  compile_ok=True  import_compl=1.0
  vue-tsc returncode=0, stdout=, stderr=
pretty/03 [a/-/a/-]  parse_ok=True  compile_ok=True  import_compl=1.0
  vue-tsc returncode=0, stdout=, stderr=
pretty/04 [a/-/a/-]  parse_ok=True  compile_ok=True  import_compl=1.0
  vue-tsc returncode=0, stdout=, stderr=
pretty/05 [a/-/a/-]  parse_ok=True  compile_ok=True  import_compl=1.0
  vue-tsc returncode=2, stdout=src/_eval_tmp/CompileCheck.vue:58:22 - error TS2322: Type 'string | ((...args: any) => string) | undefined' is not assignable to type 'string | undefined'.
  Type '(...args: any) => string' is not assignable to type 'string'.

58             <Button :label="item.label" severity="secondary" outlined class="w-full !justify-start" />
                        ~~~~~

  node_modules/primevue/button/index.d.ts:97:5
   

## 10. Aggregation (Macro- / Micro-Averages, grouped by approach, prompt strategy, complexity)

In [25]:
def group_key(r: dict) -> tuple:
    return (r['approach'], r['strategy'], r['model'], r['prompt_strategy'])


def group_label(key: tuple) -> str:
    approach, strategy, model, prompt_strategy = key
    base = strategy if not model else f'{strategy}_{model}'

    # prompt_strategy is genuinely None for approach 'a' (no artificial
    # placeholder label) -- omit the suffix entirely instead of rendering
    # Python's None as the literal string 'None' in printed output.
    return f'{base}__{prompt_strategy}' if prompt_strategy else base


def rate(items: list[dict], field: str) -> float | None:
    """Share of True among all defined (non-None) values of a binary field."""
    vals = [i[field] for i in items if i[field] is not None]

    return sum(1 for v in vals if v) / len(vals) if vals else None


def macro_mean(items: list[dict], field: str) -> float | None:
    vals = [i[field] for i in items if i[field] is not None]

    return round(sum(vals) / len(vals), 4) if vals else None


def micro_ratio(items: list[dict], correct_field: str, total_field: str) -> float | None:
    correct = sum(i[correct_field] for i in items)
    total = sum(i[total_field] for i in items)

    return round(correct / total, 4) if total else None


def median_iqr(values: list[float]) -> dict:
    if not values:
        return {'median': None, 'q1': None, 'q3': None}

    s = sorted(values)
    n = len(s)

    def pct(p):
        idx = p * (n - 1)
        lo, hi = int(idx), min(int(idx) + 1, n - 1)
        frac = idx - lo
        return s[lo] + (s[hi] - s[lo]) * frac

    return {'median': round(pct(0.5), 2), 'q1': round(pct(0.25), 2), 'q3': round(pct(0.75), 2)}


def aggregate_uf2(items: list[dict]) -> dict:
    n = len(items)

    loc_vals = [i['loc_total'] for i in items if i['loc_total'] is not None]

    return {
        'n': n,
        'parse_ok_rate':          rate(items, 'parse_ok'),
        'compile_success_rate':   rate(items, 'compile_ok'),
        'import_completeness_macro': macro_mean(items, 'import_completeness'),

        'prop_validity_rate_micro':  micro_ratio(items, 'prop_correct', 'prop_total'),
        'binding_correctness_micro': micro_ratio(items, 'binding_correct', 'binding_total'),
        'slot_correctness_micro':    micro_ratio(items, 'slot_correct', 'slot_total'),
        'pt_correctness_micro':      micro_ratio(items, 'pt_correct', 'pt_total'),

        'avu_macro':  macro_mean(items, 'avu'),
        'cdr_macro':  macro_mean(items, 'cdr'),
        'cd_macro':   macro_mean(items, 'comment_density'),
        'loc_ratio_macro': macro_mean(items, 'loc_ratio_to_gt'),

        **{f'loc_{k}': v for k, v in median_iqr(loc_vals).items()},
    }


by_method: dict[tuple, list[dict]] = defaultdict(list)
for r in results:
    by_method[group_key(r)].append(r)

by_method_complexity: dict[tuple, list[dict]] = defaultdict(list)
for r in results:
    by_method_complexity[(group_key(r), r['complexity'])].append(r)

GROUPS = sorted(by_method.keys(), key=lambda k: (k[0], k[1], k[2] or '', k[3]))

print(f'\n{"Method":42s} {"ParseOK":>8s} {"Compile":>8s} {"ImpCompl":>9s} '
      f'{"PVR":>6s} {"Bind":>6s} {"n":>4s}')
print('-' * 90)

for key in GROUPS:
    agg = aggregate_uf2(by_method[key])

    def fmt(v):
        return f'{v:.2f}' if v is not None else '-'

    print(f'{group_label(key):42s} {fmt(agg["parse_ok_rate"]):>8s} '
          f'{fmt(agg["compile_success_rate"]):>8s} {fmt(agg["import_completeness_macro"]):>9s} '
          f'{fmt(agg["prop_validity_rate_micro"]):>6s} {fmt(agg["binding_correctness_micro"]):>6s} '
          f'{agg["n"]:4d}')


Method                                      ParseOK  Compile  ImpCompl    PVR   Bind    n
------------------------------------------------------------------------------------------
a                                              1.00     1.00      1.00   0.90   0.86   10
b1_claude-sonnet-5__few_shot                   1.00     0.60      1.00   0.71   0.77   10
b1_claude-sonnet-5__zero_shot                  1.00     1.00      1.00   0.89   0.75   10
b1_gemini-3.1-pro-preview__few_shot            1.00     1.00      1.00   0.81   0.73   10
b1_gemini-3.1-pro-preview__zero_shot           1.00     1.00      1.00   0.90   0.73   10
b1_gpt-5.6-terra__few_shot                     1.00     1.00      1.00   0.69   0.77   10
b1_gpt-5.6-terra__zero_shot                    1.00     1.00      1.00   0.71   0.76   10
b2_claude-sonnet-5__few_shot                   1.00     0.70      1.00   0.76   0.77   10
b2_claude-sonnet-5__zero_shot                  1.00     1.00      1.00   0.86   0.72   10
b2_gemin

## 11. CSV-Export

In [26]:
EVALUATIONS_DIR = Path(f'evaluations/{TYPE}')
EVALUATIONS_DIR.mkdir(parents=True, exist_ok=True)

# 1) Long format
BY_MOCKUP_CSV = EVALUATIONS_DIR / f'eval_code_quality_maintainability_{TYPE}_by_mockup.csv'
BY_MOCKUP_FIELDNAMES = [
    'mockup', 'complexity', 'index', 'approach', 'strategy', 'prompt_strategy', 'model', 'run',
    'parse_ok', 'compile_ok', 'import_completeness',
    'prop_correct', 'prop_total', 'binding_correct', 'binding_total',
    'slot_correct', 'slot_total', 'pt_correct', 'pt_total',
    'avu', 'cdr', 'loc_template', 'loc_script', 'loc_total', 'loc_ratio_to_gt', 'comment_density',
]

with open(BY_MOCKUP_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=BY_MOCKUP_FIELDNAMES)
    writer.writeheader()
    writer.writerows(results)

print(f'Saved: {BY_MOCKUP_CSV}  ({len(results)} rows)')

# 2) Summary per configuration
SUMMARY_CSV = EVALUATIONS_DIR / f'eval_code_quality_maintainability_{TYPE}_summary.csv'
SUMMARY_FIELDNAMES = [
    'approach', 'strategy', 'prompt_strategy', 'model', 'n',
    'parse_ok_rate', 'compile_success_rate', 'import_completeness_macro',
    'prop_validity_rate_micro', 'binding_correctness_micro',
    'slot_correctness_micro', 'pt_correctness_micro',
    'avu_macro', 'cdr_macro', 'cd_macro', 'loc_ratio_macro',
    'loc_median', 'loc_q1', 'loc_q3',
]

summary_rows = []
for key in GROUPS:
    approach, strategy, model, prompt_strategy = key
    agg = aggregate_uf2(by_method[key])

    summary_rows.append({
        'approach': approach, 'strategy': strategy,
        'prompt_strategy': prompt_strategy, 'model': model or '',
        **agg,
    })

with open(SUMMARY_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=SUMMARY_FIELDNAMES)
    writer.writeheader()
    writer.writerows(summary_rows)

print(f'Saved: {SUMMARY_CSV}  ({len(summary_rows)} rows)')

# 3) Per configuration x complexity
BY_COMPLEXITY_CSV = EVALUATIONS_DIR / f'eval_code_quality_maintainability_{TYPE}_by_complexity.csv'
BY_COMPLEXITY_FIELDNAMES = ['approach', 'strategy', 'prompt_strategy', 'model', 'complexity'] + \
    [f for f in SUMMARY_FIELDNAMES if f not in ('approach', 'strategy', 'prompt_strategy', 'model')]

by_complexity_rows = []
for key in GROUPS:
    approach, strategy, model, prompt_strategy = key

    for group in GROUP_LEVELS:
        items = by_method_complexity[(key, group)]

        if not items:
            continue

        agg = aggregate_uf2(items)

        by_complexity_rows.append({
            'approach': approach, 'strategy': strategy, 'prompt_strategy': prompt_strategy,
            'model': model or '', 'complexity': group, **agg,  # complexity for 'components', variant for 'uis'
        })

with open(BY_COMPLEXITY_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=BY_COMPLEXITY_FIELDNAMES)
    writer.writeheader()
    writer.writerows(by_complexity_rows)

print(f'Saved: {BY_COMPLEXITY_CSV}  ({len(by_complexity_rows)} rows)')

# 4a) Degradation factor (medium/simple, hard/simple) per configuration; on
#     Parse-OK-Rate and Prop-Validity Rate as example metrics -- 'components'
#     only, since 'simple'/'medium'/'hard' is an ordered complexity
#     progression that does not apply to 'uis'.
def degradation_factor(base, target):
    if base is None or target is None or base == 0:
        return None
    return round(target / base, 4)


if TYPE == 'components':
    DEGRADATION_CSV = EVALUATIONS_DIR / f'eval_code_quality_maintainability_{TYPE}_degradation.csv'
    DEGRADATION_FIELDNAMES = [
        'approach', 'strategy', 'prompt_strategy', 'model',
        'parse_ok_simple', 'parse_ok_medium', 'parse_ok_hard',
        'degradation_parse_ok_medium', 'degradation_parse_ok_hard',
        'pvr_simple', 'pvr_medium', 'pvr_hard',
        'degradation_pvr_medium', 'degradation_pvr_hard',
    ]

    degradation_rows = []
    for key in GROUPS:
        approach, strategy, model, prompt_strategy = key

        aggs = {}
        for complexity in COMPLEXITIES:
            items = by_method_complexity[(key, complexity)]
            aggs[complexity] = aggregate_uf2(items) if items else None

        parse_ok = {c: (aggs[c]['parse_ok_rate'] if aggs[c] else None) for c in COMPLEXITIES}
        pvr = {c: (aggs[c]['prop_validity_rate_micro'] if aggs[c] else None) for c in COMPLEXITIES}

        degradation_rows.append({
            'approach': approach, 'strategy': strategy, 'prompt_strategy': prompt_strategy, 'model': model or '',
            'parse_ok_simple': parse_ok['simple'], 'parse_ok_medium': parse_ok['medium'], 'parse_ok_hard': parse_ok['hard'],
            'degradation_parse_ok_medium': degradation_factor(parse_ok['simple'], parse_ok['medium']),
            'degradation_parse_ok_hard':   degradation_factor(parse_ok['simple'], parse_ok['hard']),
            'pvr_simple': pvr['simple'], 'pvr_medium': pvr['medium'], 'pvr_hard': pvr['hard'],
            'degradation_pvr_medium': degradation_factor(pvr['simple'], pvr['medium']),
            'degradation_pvr_hard':   degradation_factor(pvr['simple'], pvr['hard']),
        })

    with open(DEGRADATION_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=DEGRADATION_FIELDNAMES)
        writer.writeheader()
        writer.writerows(degradation_rows)

    print(f'Saved: {DEGRADATION_CSV}  ({len(degradation_rows)} rows)')

else:
    # 4b) Pretty/Messy robustness ratio per configuration -- the UF5
    #     Robustheitsindex R(M,P) = Metric_messy / Metric_pretty, the 'uis'
    #     equivalent of the complexity degradation factor above.
    ROBUSTNESS_CSV = EVALUATIONS_DIR / f'eval_code_quality_maintainability_{TYPE}_robustness.csv'
    ROBUSTNESS_FIELDNAMES = [
        'approach', 'strategy', 'prompt_strategy', 'model',
        'parse_ok_pretty', 'parse_ok_messy', 'robustness_parse_ok',
        'pvr_pretty', 'pvr_messy', 'robustness_pvr',
    ]

    robustness_rows = []
    for key in GROUPS:
        approach, strategy, model, prompt_strategy = key

        aggs = {}
        for variant in VARIANTS:
            items = by_method_complexity[(key, variant)]
            aggs[variant] = aggregate_uf2(items) if items else None

        parse_ok = {v: (aggs[v]['parse_ok_rate'] if aggs[v] else None) for v in VARIANTS}
        pvr = {v: (aggs[v]['prop_validity_rate_micro'] if aggs[v] else None) for v in VARIANTS}

        robustness_rows.append({
            'approach': approach, 'strategy': strategy, 'prompt_strategy': prompt_strategy, 'model': model or '',
            'parse_ok_pretty': parse_ok['pretty'], 'parse_ok_messy': parse_ok['messy'],
            'robustness_parse_ok': degradation_factor(parse_ok['pretty'], parse_ok['messy']),
            'pvr_pretty': pvr['pretty'], 'pvr_messy': pvr['messy'],
            'robustness_pvr': degradation_factor(pvr['pretty'], pvr['messy']),
        })

    with open(ROBUSTNESS_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=ROBUSTNESS_FIELDNAMES)
        writer.writeheader()
        writer.writerows(robustness_rows)

    print(f'Saved: {ROBUSTNESS_CSV}  ({len(robustness_rows)} rows)')

Saved: evaluations\uis\eval_code_quality_maintainability_uis_by_mockup.csv  (550 rows)
Saved: evaluations\uis\eval_code_quality_maintainability_uis_summary.csv  (55 rows)
Saved: evaluations\uis\eval_code_quality_maintainability_uis_by_complexity.csv  (110 rows)
Saved: evaluations\uis\eval_code_quality_maintainability_uis_robustness.csv  (55 rows)
